# Protobuf Basics — 02: Proto Schema Definition

The .proto file is everything in Protobuf — it defines message structure,
field types, field numbers, and evolution rules.

Topics: proto3 syntax, scalar types, nested messages, repeated fields,
oneof, enums, well-known types, compiling .proto files.


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/protobuf_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('protobuf-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

## Step 1 — Proto3 Scalar Types



In [ ]:

print("""
Proto3 scalar types → Spark/Python types:
  proto3 type      wire type  Python        Spark SQL
  ─────────────────────────────────────────────────────
  double           64-bit     float         DoubleType
  float            32-bit     float         FloatType
  int32            varint     int           IntegerType
  int64            varint     int           LongType
  uint32           varint     int           LongType
  uint64           varint     int           LongType  (or DecimalType)
  sint32           varint     int           IntegerType
  sint64           varint     int           LongType
  fixed32          32-bit     int           LongType
  fixed64          64-bit     int           LongType
  bool             varint     bool          BooleanType
  string           length-del str           StringType
  bytes            length-del bytes         BinaryType

Notes:
  - int32/int64 use variable-length encoding (varint) — small values = fewer bytes
  - sint32/sint64 use ZigZag encoding — better for negative numbers
  - fixed32/fixed64 always 4/8 bytes — better for large numbers
  - No native Date/Timestamp — use google.protobuf.Timestamp (well-known type)
""")


## Step 2 — Nested Messages and Repeated Fields



In [ ]:

print("""
Proto3 schema with nested messages and repeated fields:

  syntax = "proto3";
  import "google/protobuf/timestamp.proto";

  message Address {
    string street  = 1;
    string city    = 2;
    string country = 3;
    string zip     = 4;
  }

  message LineItem {
    string sku      = 1;
    double price    = 2;
    int32  quantity = 3;
  }

  message Order {
    int64     order_id    = 1;
    int64     customer_id = 2;
    string    product     = 3;
    string    category    = 4;
    Address   address     = 5;   // nested message
    repeated LineItem items = 6; // array of messages
    repeated string tags    = 7; // array of scalars
    OrderStatus status      = 8; // enum
    google.protobuf.Timestamp created_at = 9; // well-known timestamp

    enum OrderStatus {
      PENDING   = 0;
      CONFIRMED = 1;
      SHIPPED   = 2;
      DELIVERED = 3;
      CANCELLED = 4;
    }
  }

Mapping to Spark:
  nested message → StructType
  repeated field → ArrayType
  enum           → IntegerType (the enum integer value)
  Timestamp      → StructType(seconds: long, nanos: int) or TimestampType
""")


## Step 3 — oneof: Union Fields



In [ ]:

print("""
oneof: at most one field in a group can be set

  message PaymentMethod {
    oneof method {
      CreditCard  credit_card  = 1;
      BankTransfer bank_transfer = 2;
      Crypto      crypto        = 3;
    }
  }

  message CreditCard { string card_number = 1; string expiry = 2; }
  message BankTransfer { string iban = 1; string bic = 2; }
  message Crypto { string wallet = 1; string currency = 2; }

In Spark:
  oneof maps to a struct with all fields nullable.
  Only one field will be non-null at any time.
  Checking which variant: filter on non-null field.

  df.filter(F.col("payment.credit_card").isNotNull())  // credit card payments
  df.filter(F.col("payment.crypto").isNotNull())        // crypto payments
""")


## Step 4 — Field Numbers: The Evolution Contract



In [ ]:

print("""
Field numbers are the key to Protobuf schema evolution.
They are permanently assigned — you can NEVER reuse a field number.

  message Order {
    int64  order_id    = 1;  // ← field number 1, always means order_id
    string product     = 2;  // ← field number 2, always means product
    double price       = 3;  // ← field number 3, always means price
    // field 4 was removed — RESERVE it to prevent reuse:
    reserved 4;
    reserved "old_discount";  // also reserve the name
    string status      = 5;  // ← new field (was field 6 before)
  }

Rules:
  ✅ Add new fields (any unused number) — old data has zero/empty for new field
  ✅ Remove fields — MUST reserve the number to prevent future reuse
  ❌ Change field number — breaks all existing data
  ❌ Change field type incompatibly (int32 → string)
  ✅ Rename field — field number stays the same, binary is unaffected
  ✅ Change int32 → int64 (wire compatible)
  ✅ Change string → bytes (wire compatible)
""")


## Step 5 — Compiling .proto Files



In [ ]:

import subprocess, pathlib, os

PROTO_DIR = f"{DATA_DIR}/protos"
pathlib.Path(PROTO_DIR).mkdir(parents=True, exist_ok=True)

# Write .proto file
proto_content = """syntax = "proto3";
package example;

message Order {
  int64  order_id    = 1;
  int64  customer_id = 2;
  string product     = 3;
  string category    = 4;
  string country     = 5;
  int32  quantity    = 6;
  double price       = 7;
  double revenue     = 8;
  string status      = 9;
}

message OrderBatch {
  repeated Order orders = 1;
  int64 batch_id         = 2;
  string created_at      = 3;
}
"""

proto_file = f"{PROTO_DIR}/orders.proto"
pathlib.Path(proto_file).write_text(proto_content)
print(f"Written: {proto_file}")

# Try to compile with protoc
result = subprocess.run(
    ["protoc", f"--python_out={PROTO_DIR}", f"--proto_path={PROTO_DIR}",
     f"--descriptor_set_out={PROTO_DIR}/orders.desc", "--include_imports", proto_file],
    capture_output=True, text=True
)
if result.returncode == 0:
    desc_size = pathlib.Path(f"{PROTO_DIR}/orders.desc").stat().st_size
    print(f"✅ Compiled to orders.desc ({desc_size} bytes)")
    py_file = f"{PROTO_DIR}/orders_pb2.py"
    if pathlib.Path(py_file).exists():
        print(f"✅ Generated orders_pb2.py")
else:
    print(f"protoc not available: {result.stderr[:100]}")
    print("Install: apt-get install protobuf-compiler")
    print("Or: pip install grpcio-tools")
    print("Continuing with manual serialization simulation...")


## Summary



In [ ]:

print("""
Proto3 cheat sheet:
  scalar: double|float|int32|int64|uint32|uint64|bool|string|bytes
  nested: message Name { ... } then FieldType field_name = N;
  array:  repeated FieldType field_name = N;
  union:  oneof name { TypeA a = 1; TypeB b = 2; }
  enum:   enum Name { VALUE_ZERO = 0; VALUE_ONE = 1; }

Field number rules (CRITICAL):
  - Assign once, never change
  - Never reuse a removed number (use: reserved 4;)
  - Numbers 1-15: 1 byte in wire format (use for frequent fields)
  - Numbers 16-2047: 2 bytes in wire format

Compiling:
  protoc --python_out=. --descriptor_set_out=schema.desc orders.proto
  # schema.desc needed by Spark from_protobuf()
""")
